## 04 - Impact of Room Type on Price

- **Research Question:** How much does the type of accommodation (`room_type`) affect listing prices? How much cheaper is a private room compared to an entire apartment (the "privacy discount"), and is this pattern consistent across different cities?
- **Variables:**
  - `room_type`
  - `price`
- **Statistical Tools:**
  - **Conditional Statistics:** Calculating the sample mean and sample median for each specific room type in every city.
  - **The Privacy Discount:** Measuring how much cheaper private rooms are compared to entire homes.
  - **Hypothesis Testing:**
    - **One-way ANOVA (F-test):** Checking if average prices differ significantly across all room categories.
    - **Welch's t-test and Mann-Whitney U test:** Comparing prices directly between `Entire home/apt` and `Private room` using both parametric and non-parametric tests to handle skewed data.
  - **Visualizations:**
    - Multi-panel side-by-side boxplots showing price spreads and medians across room types for each city.
    - Comparative charts illustrating how privacy discounts vary across global locations.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from src.cities import CITIES
from src.load import load_all

cols = ['id', 'price', 'room_type', 'latitude', 'longitude', 'neighbourhood_cleansed']
data = load_all(usecols=cols)

data = data.dropna(subset=['price']).copy()

data['city_name'] = data['city'].map({k: v.name for k, v in CITIES.items()})
data['currency'] = data['city'].map({k: v.currency for k, v in CITIES.items()})

fx_to_eur = {
    'EUR': 1.00,
    'USD': 0.86,
    'JPY': 0.0055,
    'MXN': 0.05
}
data['fx_to_eur'] = data['currency'].map(fx_to_eur)
data['price_eur'] = data['price'] * data['fx_to_eur']

num_offers = len(data)
num_cities = data['city'].nunique()
print(f"Loaded {num_offers:,} listings with valid price and room_type across {num_cities} cities.")

In [ ]:
records = []
room_order = ['Entire home/apt', 'Private room', 'Hotel room', 'Shared room']

for city_key, city_obj in CITIES.items():
    city_df = data[data['city'] == city_key]
    total_city = len(city_df)
    
    for r_type in room_order:
        sub = city_df[city_df['room_type'] == r_type]['price']
        if len(sub) > 0:
            records.append({
                'City': city_obj.name,
                'Currency': city_obj.currency,
                'Room Type': r_type,
                'Listings Count': len(sub),
                'Market Share (%)': round((len(sub) / total_city) * 100, 1),
                'Median Price': round(sub.median(), 2),
                'Mean Price': round(sub.mean(), 2),
                'Std Dev (s)': round(sub.std(), 2),
                'IQR': round(sub.quantile(0.75) - sub.quantile(0.25), 2)
            })

#Conditional Price Statistics by Room Type
df_room_stats = pd.DataFrame(records)
df_room_pivot = df_room_stats.pivot(index=['City', 'Currency'], columns='Room Type', values=['Listings Count', 'Median Price', 'Mean Price'])
df_room_pivot

In [ ]:
# Statistical Hypothesis Testing: ANOVA, Welch's t-test, and Mann-Whitney U test
test_results = []

for city_key, city_obj in CITIES.items():
    city_df = data[data['city'] == city_key]
    
    # 1. One-way ANOVA across all room types
    groups = [group['price'].values for _, group in city_df.groupby('room_type')]
    f_stat, p_anova = stats.f_oneway(*groups)
    
    # 2. Entire home vs Private room comparisons
    entire = city_df[city_df['room_type'] == 'Entire home/apt']['price']
    private = city_df[city_df['room_type'] == 'Private room']['price']
    
    med_entire = entire.median()
    med_private = private.median()
    discount_pct = (1 - med_private / med_entire) * 100
    
    # Parametric Welch's t-test (unequal variances)
    t_stat, p_ttest = stats.ttest_ind(entire, private, equal_var=False)
    
    # Non-parametric Mann-Whitney U test
    u_stat, p_mwu = stats.mannwhitneyu(entire, private, alternative='two-sided')
    
    test_results.append({
        'City': city_obj.name,
        'Currency': city_obj.currency,
        'Median Entire': round(med_entire, 2),
        'Median Private': round(med_private, 2),
        'Private Discount (%)': round(discount_pct, 1),
        'ANOVA F-stat': round(f_stat, 1),
        'ANOVA p-val': '< 0.0001' if p_anova < 0.0001 else f'{p_anova:.4f}',
        'Welch t-stat': round(t_stat, 1),
        'Welch p-val': '< 0.0001' if p_ttest < 0.0001 else f'{p_ttest:.4f}',
        'Mann-Whitney U': f'{u_stat:,.0f}',
        'MWU p-val': '< 0.0001' if p_mwu < 0.0001 else f'{p_mwu:.4f}'
    })

# Statistical Significance of Price Differences by Room Type
df_tests = pd.DataFrame(test_results).set_index('City')
df_tests

In [ ]:
# Side-by-Side Boxplots by Room Type Across All 6 Cities
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

colors_room = {
    'Entire home/apt': '#1d3557',
    'Private room': '#457b9d',
    'Hotel room': '#f4a261',
    'Shared room': '#e63946'
}

for idx, (city_key, city_obj) in enumerate(CITIES.items()):
    ax = axes[idx]
    city_df = data[data['city'] == city_key]
    
    # Clip at 98th percentile for clear box and whisker display
    p98 = city_df['price'].quantile(0.98)
    city_vis = city_df[city_df['price'] <= p98]
    
    sns.boxplot(
        data=city_vis, x='room_type', y='price', order=room_order,
        hue='room_type', legend=False,
        palette=colors_room, ax=ax, width=0.55, showfliers=False
    )
    
    # Annotate medians
    medians = city_df.groupby('room_type')['price'].median().reindex(room_order)
    for r_idx, (r_type, med) in enumerate(medians.items()):
        if pd.notna(med):
            ax.text(r_idx, med + p98 * 0.02, f'{med:,.0f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
            
    groups = [g['price'].values for _, g in city_df.groupby('room_type')]
    f_stat, p_val = stats.f_oneway(*groups)
    
    ax.set_title(f"{city_obj.name} ({city_obj.currency})\nANOVA: F = {f_stat:.1f} (p < 0.0001***)", fontsize=11, fontweight='bold', pad=8)
    ax.set_ylabel(f"Price ({city_obj.currency})", fontsize=10)
    ax.set_xlabel('')
    ax.set_xticks(range(len(room_order)))
    ax.set_xticklabels(['Entire', 'Private', 'Hotel', 'Shared'], fontsize=9, fontweight='bold')

plt.suptitle("Figure 1: Price Boxplots by Room Type Across All 6 Cities\nDrastic Price Drop from Entire Homes to Private & Shared Rooms", 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/study4_room_type_boxplots_by_city.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# The Privacy Premium and Cross-City Room Type Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

discount_data = []
for city_key, city_obj in CITIES.items():
    city_df = data[data['city'] == city_key]
    med_entire = city_df[city_df['room_type'] == 'Entire home/apt']['price'].median()
    med_priv = city_df[city_df['room_type'] == 'Private room']['price'].median()
    discount = (1 - med_priv / med_entire) * 100
    discount_data.append({'City': city_obj.name, 'Discount': discount})

df_disc = pd.DataFrame(discount_data).sort_values('Discount', ascending=False)
bars = ax1.bar(df_disc['City'], df_disc['Discount'], color='#1d3557', width=0.55)
ax1.set_title('Private Room Discount vs Entire Home (Median %)', fontsize=12, fontweight='bold', pad=10)
ax1.set_ylabel('Discount (%) [1 - Median(Private) / Median(Entire)]', fontsize=10)
ax1.set_ylim(0, 85)
ax1.tick_params(axis='x', rotation=15)

for bar in bars:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 1.5, f"{h:.1f}%", ha='center', va='bottom', fontsize=9.5, fontweight='bold')

city_order = ['Barcelona', 'Los Angeles', 'Madrid', 'New York', 'Mexico City', 'Tokyo']
sub_data = data[data['room_type'].isin(['Entire home/apt', 'Private room'])].copy()

sns.boxplot(
    data=sub_data, x='city_name', y='price_eur', hue='room_type', order=city_order,
    palette={'Entire home/apt': '#1d3557', 'Private room': '#457b9d'}, ax=ax2,
    showfliers=False, width=0.65
)
ax2.set_ylim(0, 450)
ax2.set_title('Price Comparison: Entire Home vs Private Room (EUR Benchmark)', fontsize=12, fontweight='bold', pad=10)
ax2.set_ylabel('Price (EUR equivalent)', fontsize=10)
ax2.set_xlabel('')
ax2.tick_params(axis='x', rotation=15)
ax2.legend(title='Room Type', loc='upper right', frameon=True)

plt.suptitle("Figure 2: The Privacy Premium in Airbnb Pricing Across Global Cities", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/study4_privacy_premium_and_discount.png', dpi=300, bbox_inches='tight')
plt.show()

## Findings

### 1. The Massive "Privacy Premium" Across Global Destinations
- Accommodation type (`room_type`) is one of the strongest factors determining listing prices in every city.
- **Renting an entire property is significantly more expensive than a private room:**
  - **Barcelona:** Median price drops from €221.50 (Entire) to €64.00 (Private) - a 71.1% discount.
  - **Los Angeles:** Median price drops from $282.00 to $85.37 - a 69.7% discount.
  - **Madrid:** Median price drops from €144.00 to €64.00 - a 55.6% discount.
  - **Mexico City:** Median price drops from $2,163 MXN to $970 MXN - a 55.2% discount.
  - **New York:** Median price drops from $222.87 to $102.97 - a 53.8% discount.
  - **Tokyo:** Median price drops from ¥20,537 to ¥15,000 - a 27.0% discount.

### 2. Cross-City Variations and Housing Culture
- **Highest discount in leisure/tourist hubs:** Barcelona (71.1%) and Los Angeles (69.7%) show the largest savings for private rooms, as tourists in these cities strongly prefer whole apartments or beachfront homes.
- **Why Tokyo is an exception (27.0% discount):** Apartments in Tokyo are naturally compact and micro-sized. Because an "entire apartment" in Tokyo is often similar in square footage to a single private room, the difference in space and therefore price is much smaller than in Western cities.

### 3. Statistical Validation
- **ANOVA Test:** In all 6 cities, the test confirms (p < 0.0001) that prices vary significantly across room categories.
- **Welch's t-test and Mann-Whitney U test:** Pairwise comparisons between `Entire home/apt` and `Private room` are statistically significant (p < 0.0001) everywhere. This proves that the observed privacy discount is real and consistent across markets.

### 4. Summary and Interview Defense Key Points
- **Which statistical test is best for comparing room types?** ANOVA checks if overall differences exist across all categories. However, because price data is right-skewed, the non-parametric **Mann-Whitney U test** (or Kruskal-Wallis) is the most reliable method for comparing entire homes vs private rooms.
- **How does room type affect price dispersion?** Entire homes have a much wider price range because they include everything from small studios to luxury villas. Private rooms have much tighter, more predictable pricing.

In [ ]:
import plotly.express as px

fig = px.scatter_geo(
    data.dropna(subset=["latitude", "longitude"]),
    lat="latitude",
    lon="longitude",
    color="room_type",
    color_discrete_sequence=px.colors.qualitative.Set2,
    hover_data={"room_type": True, "price_eur": ":.1f", "neighbourhood_cleansed": True, "latitude": False, "longitude": False},
    projection="natural earth",
    opacity=0.5,
    size_max=6,
    title="Spatial Distribution of Room Types Across 6 Cities (Study 04)",
)

fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.show()

### Spatial Interpretation of Room Types
- **Spatial Segregation:** The map clearly visualizes the structural differences in accommodation types across the city. Entire homes and apartments (representing the higher price tier) cluster densely in the city center and primary tourist zones.
- **Peripheral Distribution of Private Rooms:** Private and shared rooms (budget-friendly options) show a much wider dispersion into residential and peripheral neighborhoods, which directly explains the massive "privacy premium" observed in our statistical analysis.


In [ ]:
import plotly.express as px

CITY = "Tokyo"
df = data[data["city_name"] == CITY].copy()

fig = px.scatter_map(
    df,
    lat="latitude",
    lon="longitude",
    color="room_type",
    hover_data=["room_type", "price", "neighbourhood_cleansed"],
    color_discrete_sequence=px.colors.qualitative.Set2,
    title=f"Category Map: Room Types in {CITY}",
    zoom=12,
    center={"lat": df["latitude"].median(), "lon": df["longitude"].median()},
    opacity=0.6,
    map_style="carto-positron",
)
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig.show()
